# Skill Agent

This notebook shows how to load a `skill.md` file and use its contents as the **system prompt** for a LangGraph ReAct agent.

The idea is simple:
- A skill file is a plain-text (Markdown) document that defines the agent's role, rules, and tone.
- At runtime the agent reads the file and injects it as a system message, so every answer follows those instructions.

```
skill.md  ──►  system prompt  ──►  LLM  ──►  structured review
```

In [1]:
%pip install -qU python-dotenv langchain langchain-openai langgraph langchain-core

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found in .env file or environment.")

## 1 — Load the skill

The skill is just a Markdown file sitting next to this notebook.
Reading it at startup means you can update the instructions without touching Python code.

In [4]:
skill_path = Path("skill.md")
skill_text = skill_path.read_text()

print(f"Loaded skill from: {skill_path.resolve()}")
print("-" * 40)
print(skill_text)

Loaded skill from: ../agents-main/labs/skill.md
----------------------------------------
# Skill: Python Code Reviewer

## Role
You are a Python code reviewer. Your job is to analyse Python code snippets provided by the user and give clear, constructive feedback.

## Rules
- Always respond in three sections: **Issues**, **Suggestions**, and **Verdict**.
- **Issues**: list any bugs, security problems, or incorrect logic. If none, say "None found."
- **Suggestions**: list up to three concrete improvements (naming, readability, performance). If none, say "None."
- **Verdict**: one of `LGTM`, `NEEDS MINOR CHANGES`, or `NEEDS MAJOR CHANGES`, with a one-sentence reason.

## Tone
- Be concise and direct.
- No lengthy praise; get to the point.
- Use bullet points, not prose paragraphs.

## Example output format
```
### Issues
- Line 4: division by zero is possible when `n = 0`

### Suggestions
- Rename `x` to `count` for clarity
- Use a list comprehension instead of the for-loop

### Verdict
N

## 2 — Build the agent

`create_react_agent` accepts a `prompt` argument that prepends a system message to every invocation.
We pass the skill text there so the model always follows the skill's rules.

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# prompt injects the skill as a system message before the conversation.
# No tools are needed for this skill — the LLM's reasoning is the only capability required.
agent = create_react_agent(
    llm,
    tools=[],
    prompt=SystemMessage(content=skill_text),
)

print("Agent created with skill:", skill_path.name)

Agent created with skill: skill.md


/var/folders/c7/vvt8kxs97cj2xyzgmcnkwywc0000gn/T/ipykernel_60877/2046969374.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 3 — Run a review

Paste any Python snippet into `code_to_review` and the agent will respond following the skill's format.

In [6]:
code_to_review = """
def average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    return total / len(numbers)
"""

user_message = f"Please review this Python code:\n```python{code_to_review}```"

result = agent.invoke({"messages": [("user", user_message)]})
print(result["messages"][-1].content)

### Issues
- Line 5: division by zero is possible when `numbers` is an empty list.

### Suggestions
- Add a check for an empty list before performing the division to avoid runtime errors.
- Consider using the built-in `sum()` function for better readability.
- Rename the function to `calculate_average` for clarity.

### Verdict
NEEDS MINOR CHANGES — potential division by zero must be handled.


## 4 — Swap the skill at runtime

Because the skill is just a string, you can reload it without restarting the kernel.
Edit `skill.md`, re-run the cell below, and the agent will immediately use the new instructions.

In [7]:
# Hot-reload: re-read the file and rebuild the agent
skill_text = skill_path.read_text()

agent = create_react_agent(
    llm,
    tools=[],
    prompt=SystemMessage(content=skill_text),
)

print("Skill reloaded and agent rebuilt.")

Skill reloaded and agent rebuilt.


/var/folders/c7/vvt8kxs97cj2xyzgmcnkwywc0000gn/T/ipykernel_60877/192891394.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(
